In [6]:
import random
from pokerkit import *
from dotenv import load_dotenv
import json
from anthropic import Anthropic, beta_tool
from mediumterm import MediumTermMemory
from longterm import LongTermMemory
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
GAME_ID = "game_001"   #!this NEEDS TO BE CHANGED MANUALLY EACH RUN FOR LONG TERM MEMORY

In [3]:
load_dotenv()

client = Anthropic()

In [4]:
PLAYER_LABEL_MAP = {1: "P2", 2: "P3"}  #our opponents, for example seat 1 has P2, logged as P2.txt


def build_system_prompt(ltm: LongTermMemory, active_player_ids: list[str]) -> str:
    """Construct the system prompt, injecting LTM profiles for known opponents."""

    base = (
        "You are a poker agent playing 3-handed Texas Hold'em No-Limit.\n"
        "Blinds are 1000/2000. You are Player 1 (Hero). Player 2 is P2, Player 3 is P3 and so on.\n"
        "Think about pot odds, position, and hand strength before deciding. "
        "Make sure to carefully read and use the information in the provided json.\n"
        "If you are unsure of your hand strength, use the get_equity_strength tool to calculate your win probability.\n"
        "Remember that the equity tool only tells you strength of the cards, not the behavior of your opponents.\n"
        "Use self-notes to reflect on your past actions, and opponent notes to reflect on the behavior of your opponents.\n"
        "Use the take_action tool to submit your decision."
    )

    self_profile = ltm.read_self_profile()
    ltm_section = (
        "\n\n--- YOUR STRATEGIC SELF-NOTES (from long-term memory) ---\n"
        + self_profile
        + "\n--- END SELF NOTES ---")

    opponent_sections = []
    for pid in active_player_ids:
        profile = ltm.lookup_player(pid)
        opponent_sections.append(f"\n--- LONG-TERM PROFILE: {pid} ---\n{profile}\n--- END PROFILE: {pid} ---")
    if opponent_sections:
        ltm_section += "\n" + "\n".join(opponent_sections)
    return base + ltm_section


def extract_tags_from_reasoning(reasoning_texts: list[str]) -> dict[str, list[str]]:
    tag_map: dict[str, list[str]] = {}

    for text in reasoning_texts:
        if not text:
            continue
        words = text.split()
        for i, word in enumerate(words):
            player_label = word.strip(",:").upper()
            if not (player_label.startswith("P") and player_label[1:].isdigit()):
                continue
            if player_label[1] not in "123456789": #! we only support opponent names P1 through P9
                continue
            #search for rating
            lookahead = words[i+1 : i+11]
            for chunk in lookahead:
                if "/10" not in chunk:
                    continue
                score_part = chunk.split("/10")[0].strip()
                if not score_part.isdigit():
                    break
                score = int(score_part)
                if score >= 7:
                    tag = "aggressive"
                elif score <= 3:
                    tag = "passive"
                else:
                    tag = "balanced"
                tag_map.setdefault(player_label, []).append(tag)
                break
    return {pid: [Counter(tags).most_common(1)[0][0]] for pid, tags in tag_map.items()}

def run_post_hand_reflection(ltm: LongTermMemory, eval_text: str, active_player_ids: list[str], reasonings: list, payoffs: tuple) -> None:
    """
    Called once after each hand to:
    1. Write per-opponent observations to LTM.
    2. Append a self-note about how Hero played.
    """
    if not eval_text: return

    reasoning_texts = [r[2] for r in reasonings if len(r) >= 3]
    tag_map = extract_tags_from_reasoning(reasoning_texts)
    for pid in active_player_ids:
        tags = tag_map.get(pid, [])
        ltm.log_player_update(player_id=pid, observation=eval_text, tags=tags)
    hero_payoff = payoffs[0] if payoffs else 0
    outcome = "won" if hero_payoff > 0 else ("lost" if hero_payoff < 0 else "broke even")
    hero_reasoning = " | ".join(f"{reasoning[0]} {reasoning[1] if reasoning[1] else ''}: {reasoning[2]}"
        for reasoning in reasonings if len(reasoning) >= 3)
    self_note = (
        f"Hand outcome: {outcome} ({hero_payoff:+,} chips). "
        f"Opponent eval: {eval_text[:300]}. "
        f"Hero reasoning summary: {hero_reasoning[:400]}."
    )
    ltm.log_self_update(observation=self_note, section="Lessons learned")


@beta_tool
def take_action(action: str, reasoning: str, amount: float = 0) -> str:
    """Submit your poker action for this turn and give a maximum 3 sentence explanation for why.

    Args:
        action: One of fold, check, call, raise
        amount: Raise amount in chips. Required if action is raise.
        reasoning: Reasoning for why you took a poker action. Maximum 3 sentences. Rate how aggressive each opponent's actions are on a 1-10 scale, but only if you made it to the flop or later. 
    """
    return json.dumps({"status": "accepted", "action": action, "amount": amount, 'reasoning': reasoning})

@beta_tool
def get_equity_strength(hole_cards: str, board_cards: str, active_players: int) -> str:
    """Calculates the mathematical win probability (equity) of your current hand using Monte Carlo simulations.
    Call this tool when you are facing a difficult decision, a large bet, or are unsure of your exact hand strength.
    
    Args:
        hole_cards: Your two hole cards as a string (e.g., 'AsKc').
        board_cards: The visible board cards as a string (e.g., 'Td9h2c'). Leave empty '' for preflop.
        active_players: Total number of players dealt into the simulation.
    """
    
    print(f"\n   [TOOL TRIGGERED] Calculating equity for {hole_cards} | Board: {board_cards} | Players: {active_players}")
    
    board_input = tuple(Card.parse(board_cards)) if board_cards else ()
    
    calculated_equity = calculate_hand_strength(
        active_players,      
        parse_range(hole_cards),
        board_input,
        active_players,      
        5,                
        Deck.STANDARD,
        (StandardHighHand,),
        sample_count=500
    )
    
    win_percentage = round(calculated_equity * 100, 1)
    
    return json.dumps({
        "status": "success", 
        "equity": calculated_equity,
        "message": f"You have a {win_percentage}% chance of winning."
    })


def run_turn(state_json, ltm: LongTermMemory, active_player_ids: list[str]):
    system_prompt = build_system_prompt(ltm, active_player_ids)

    messages = [{
        "role": "user",
        "content": f"""It's your turn to act. Here is the current game state:
    
    ```json
    {state_json}
    ```
    
    Decide your action."""}]

    runner = client.beta.messages.tool_runner(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        tools=[take_action, get_equity_strength],
        system=system_prompt,
        messages=messages
    )

    agent_action = None
    tokens = ""

    for message in runner:
        tokens = f"Input: {message.usage.input_tokens}, Output: {message.usage.output_tokens}"
        for block in message.content:
            if block.type == "tool_use" and block.name == "take_action":
                agent_action = block.input

    return agent_action, tokens

def street_converter(index):
    if index == 0:
        return 'preflop'
    elif index == 1:
        return 'flop'
    elif index == 2:
        return 'turn'
    elif index == 3:
        return 'river'
    
def get_visible_operations(state):
    visible = []
    showdown = sum(1 for p in state.statuses if p) > 1
    winners = set()
    for op in state.operations:
        if isinstance(op, ChipsPushing):
            winners = {i for i, amt in enumerate(op.amounts) if amt > 0}

    for op in state.operations:
        if isinstance(op, HoleDealing):
            continue
        elif isinstance(op, CardBurning):
            continue
        elif isinstance(op, HoleCardsShowingOrMucking):
            if showdown or op.player_index in winners:
                visible.append(op)
        else:
            visible.append(op)
    return visible

judge_system_prompt = """You are an evaluating agent for players playing Texas Hold-em. Your job is to evaluate and provide useful information about the other non-agent players to aid Player 1 in defeating them. Do not evaluate Player 1 (Hero), focus on Player 2 (P2), and Player 3 (P3). Remember that index 0 is the agent, index 1 is Player 2 and so on. Use the generate_observation tool to submit your decision."""

@beta_tool
def generate_observation(evaluation: str) -> str:
    """Evaluate the action history of the game in the previous poker hand. 

    Args:
        evaluation: The evaluation of how the other players played. Be specific as to any mistakes or good moves each non-Hero player made. 
    """
    return json.dumps({"status": "accepted", "evaluation": evaluation})

    
def run_observation_generator(action_history):
    eval_json = {'action_history': action_history}
    messages = [{
        "role": "user",
        "content": f"""Here is the action history of the game. 
    
    ```json
    {eval_json}
    ```
    
    Generate your observations."""}]

    runner = client.beta.messages.tool_runner(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        tools=[generate_observation],
        system=judge_system_prompt,
        messages=messages
    )

    agent_action = None

    for message in runner:
        for block in message.content:
            if block.type == "tool_use" and block.name == "generate_observation":
                agent_action = block.input["evaluation"]

    return agent_action


In [ ]:
def pocket_pair_strategy(state, player_index):
    hole = state.hole_cards[player_index]
    is_pair = len(hole) >= 2 and hole[0].rank == hole[1].rank
    strong_pair_ranks = {"A", "K", "Q", "J", "T", "9", "8", "7"}
    is_strong_pair = is_pair and hole[0].rank in strong_pair_ranks
    if is_strong_pair and state.can_complete_bet_or_raise_to():
        min_raise = state.min_completion_betting_or_raising_to_amount
        max_raise = state.max_completion_betting_or_raising_to_amount
        target_raise = min(min_raise * 2, max_raise)
        state.complete_bet_or_raise_to(target_raise)
    elif is_pair and state.can_check_or_call():
        state.check_or_call()
    elif state.can_fold():
        state.fold()
    else:
        state.check_or_call()

def punter_strategy(state, player_index):
    punter_is_punting_this_hand = random.random() < 0.1
    if not punter_is_punting_this_hand:
        if state.can_fold():
            state.fold()
        else:
            state.check_or_call()
        return
    if state.can_complete_bet_or_raise_to():
        min_raise = state.min_completion_betting_or_raising_to_amount
        max_raise = state.max_completion_betting_or_raising_to_amount
        target_raise = min(min_raise * 2, max_raise)
        state.complete_bet_or_raise_to(target_raise)
    elif state.can_check_or_call():
        state.check_or_call()
    elif state.can_fold():
        state.fold()

def tight_passive_strategy(state, player_index):
    r = random.random()
    if r < 0.9 and state.can_fold():
        state.fold()
    elif state.can_check_or_call():
        state.check_or_call()
    elif state.can_fold():
        state.fold()

def random_raiser(state, player_index):
    action_rand = random.random()
    if action_rand < 0.1 and state.can_complete_bet_or_raise_to():
    # 10% chance to raise to 3x the big blind
        min_raise = state.min_completion_betting_or_raising_to_amount
        max_raise = state.max_completion_betting_or_raising_to_amount # All-in

    # Raise to 2x the minimum required to keep it moving
        target_raise = min(min_raise * 2, max_raise)
        state.complete_bet_or_raise_to(target_raise)
    elif action_rand < 0.2 and state.can_fold():
    # 10% chance to fold
        state.fold()
    else:
        state.check_or_call()

opponent_strategies = {
    1: pocket_pair_strategy,
    2: punter_strategy,
    3: tight_passive_strategy,
    4: random_raiser}

In [5]:
med = MediumTermMemory()
med.new_game(game_id=GAME_ID, players=['Hero', 'P2', 'P3', 'P4', 'P5'])
ltm = LongTermMemory()
ltm.new_session(game_id=GAME_ID)

# Initial Setup
player_stacks = [10_000, 10_000, 10_000]
hand_count = 0


# The Global Game Loop: Runs until only one player has chips
while len([s for s in player_stacks if s > 0]) > 1:
    reasonings = []
    active_indices = [i for i, stack in enumerate(player_stacks) if stack > 0]

    if len(active_indices) < 2:
        break

    # Map engine seat indices → human player IDs for LTM lookups
    # Seat 0 = Hero; seat 1+ = opponents in position order
    active_player_ids = [
        PLAYER_LABEL_MAP[i] for i in active_indices if i in PLAYER_LABEL_MAP
    ]

    current_stacks = tuple(player_stacks[i] for i in active_indices)
    hand_count += 1
    print(f"\n--- STARTING HAND {hand_count} ---")

    state = NoLimitTexasHoldem.create_state(
        (
            Automation.ANTE_POSTING,
            Automation.BET_COLLECTION,
            Automation.BLIND_OR_STRADDLE_POSTING,
            Automation.CARD_BURNING,
            Automation.HOLE_DEALING,
            Automation.BOARD_DEALING,
            Automation.HOLE_CARDS_SHOWING_OR_MUCKING,
            Automation.HAND_KILLING,
            Automation.CHIPS_PUSHING,
            Automation.CHIPS_PULLING,
        ),
        True, 0, (1000, 2000), 2000, tuple(current_stacks), len(active_indices)
    )

    # Per-Turn Decision Loop
    while state.status:
        if state.can_deal_hole():
            state.deal_hole()
        elif state.can_deal_board():
            state.deal_board()

        elif state.actor_index is not None:
            res = get_visible_operations(state)
            obs = {
                "your_index": 0,
                "pot": state.total_pot_amount,
                "board": state.board_cards,
                "hole": state.hole_cards[state.actor_index],
                "stacks": state.stacks,
                "bets": state.bets,
                "street": street_converter(state.street_index),
                "can_fold?": state.can_fold(),
                "can_check_or_call?": state.can_check_or_call(),
                "can_raise?": state.can_complete_bet_or_raise_to(),
                "min_raise": state.min_completion_betting_or_raising_to_amount,
                "max_raise": state.max_completion_betting_or_raising_to_amount,
                "how_much_to_call": state.checking_or_calling_amount,
                "action_history": res
            }

            if state.actor_index == 0:#hero/agent
                action = run_turn(obs, ltm, active_player_ids)
                if action[0]['action'] in ['check', 'call']:
                    state.check_or_call()
                elif action[0]['action'] == 'raise':
                    state.complete_bet_or_raise_to(action[0]['amount'])
                elif action[0]['action'] == 'fold':
                    state.fold()
                try:
                    reasonings.append([action[0]['action'], action[0]['amount'], action[0]['reasoning']])
                except:
                    reasonings.append([action[0]['action'], 0, action[0]['reasoning']])

            else:  # Random bots
                global_player_index = active_indices[state.actor_index]
                opponent_strategies[global_player_index](state, state.actor_index)
        else:
            break

   
    med.ingest_hand(action_history=list(map(lambda x: str(x), get_visible_operations(state))), reasoning=reasonings,
        chip_changes=state.payoffs)
    eval_text = run_observation_generator(get_visible_operations(state))
    med.log_trend(eval_text)

    run_post_hand_reflection(ltm=ltm, eval_text=eval_text, active_player_ids=active_player_ids, reasonings=reasonings, payoffs=state.payoffs)
    for i, global_idx in enumerate(active_indices):
        player_stacks[global_idx] += int(state.payoffs[i])
    
    print(f"Hand {hand_count} Results: {state.payoffs}")
    print(f"New Stacks: {player_stacks}")

ltm.close_session()#flush to disk.

print(f"\nGAME OVER after {hand_count} hands!")
print(f"Final Winner Stacks: {player_stacks}")
print(f"\nLTM index now contains: {list(ltm._index.keys())}")



--- STARTING HAND 1 ---

   [🔍 TOOL TRIGGERED] Calculating equity for QdJc9c | Board: AhJd8c | Players: 3

   [🔍 TOOL TRIGGERED] Calculating equity for Qd9c | Board: AhJd8c | Players: 3

   [🔍 TOOL TRIGGERED] Calculating equity for QdJc | Board: AhJd8c | Players: 3

   [🔍 TOOL TRIGGERED] Calculating equity for Qd9c | Board: AhJd8c | Players: 3
Hand 1 Results: [-2000, -6000, 8000]
New Stacks: [8000, 4000, 18000]

--- STARTING HAND 2 ---
Hand 2 Results: [-1000, 5000, -4000]
New Stacks: [7000, 9000, 14000]

--- STARTING HAND 3 ---

   [🔍 TOOL TRIGGERED] Calculating equity for 8sKd | Board: 9s8hJc | Players: 3

   [🔍 TOOL TRIGGERED] Calculating equity for 8sKd | Board: 9s8hJc | Players: 3
Hand 3 Results: [-2000, -2000, 4000]
New Stacks: [5000, 7000, 18000]

--- STARTING HAND 4 ---
Hand 4 Results: [-1000, -2000, 3000]
New Stacks: [4000, 5000, 21000]

--- STARTING HAND 5 ---

   [🔍 TOOL TRIGGERED] Calculating equity for 3d3c | Board: 9dAsTd | Players: 3

   [🔍 TOOL TRIGGERED] Calculating eq